# RAG 应用程序

![Simple RAG](../../images/simple_rag.png)

在这个笔记本中，我们将设置一个简单的 RAG 应用程序，在学习更多关于 LangSmith 的知识时会用到它。

RAG（检索增强生成）是一种流行的技术，用于为大语言模型提供相关文档，使它们能够更好地回答用户的问题。

在我们的案例中，我们将索引一些 LangSmith 文档！

LangSmith 让跟踪任何大语言模型应用程序变得简单，不需要 LangChain！

### 设置

请确保设置您的环境变量，包括您的 OpenAI API 密钥。

In [ ]:
# 您可以在代码中直接设置！
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"

In [ ]:
# 或者您可以使用 .env 文件
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

### 简单的 RAG 应用程序

In [ ]:
from langsmith import traceable
from openai import OpenAI
from typing import List
import nest_asyncio
from utils import get_vector_db_retriever

MODEL_PROVIDER = "openai"
MODEL_NAME = "gpt-4o-mini"
APP_VERSION = 1.0
RAG_SYSTEM_PROMPT = """您是一个问答任务的助手。
使用以下检索到的上下文片段来回答对话中的最新问题。
如果您不知道答案，请直接说您不知道。
最多使用三句话，保持答案简洁。
"""

openai_client = OpenAI()
nest_asyncio.apply()
retriever = get_vector_db_retriever()

"""
retrieve_documents
- 根据用户的问题从向量存储中返回获取的文档
"""
@traceable(run_type="chain")
def retrieve_documents(question: str):
    return retriever.invoke(question)

"""
generate_response
- 在格式化输入后调用 `call_openai` 来生成模型响应
"""
@traceable(run_type="chain")
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    messages = [
        {
            "role": "system",
            "content": RAG_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"上下文: {formatted_docs} \n\n 问题: {question}"
        }
    ]
    return call_openai(messages)

"""
call_openai
- 从 OpenAI 返回聊天完成输出
"""
@traceable(run_type="llm")
def call_openai(
    messages: List[dict], model: str = MODEL_NAME, temperature: float = 0.0
) -> str:
    return openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )

"""
langsmith_rag
- 调用 `retrieve_documents` 来获取文档
- 调用 `generate_response` 来基于获取的文档生成响应
- 返回模型响应
"""
@traceable(run_type="chain")
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content

这应该需要不到一分钟的时间。我们正在 SKLearn 向量数据库中索引和存储 LangSmith 文档。

In [ ]:
question = "LangSmith 用于什么？"
ai_answer = langsmith_rag(question, langsmith_extra={"metadata": {"website": "www.google.com"}})
print(ai_answer)

### 让我们在 LangSmith 中查看一下！